# Check 3 -- do signature-derived items leak the label?

Paper mini-experiment. Every CSCAS signature is class-pure, so Base(5) drops
SignatureID. Category, ruleset, qualifiers and CVE come from the signature text
too: near-pure values of those fields would hand every miner a SignatureID
stand-in. Logic lives in `thesis.system_eval.signature_leakage`.

- **§1 Purity table** (this notebook, read-only): per value of each
  signature-derived field, on W0 and pooled over all training windows.
- **§2 Seen vs. unseen** signature recall and **§3 (a)/(b)/(c) ablation**, from
  `shell-scripts/paper/run_signature_leakage.sh` (run on the DGX, synced back).
  §2–3 don't need the §1 data load -- they can run on their own.

Windows: `baseline_split`, γ = 0.05 (W0 = CSCAS baseline train period, then
62,789-group post-split windows; the last window is test-only).

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

from thesis.pipeline.pipeline import load_or_build_alert_groups
from thesis.system_eval import signature_leakage as sl

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)


def to_booktabs(df: pd.DataFrame, caption: str, label: str, float_fmt: str = "{:.3f}") -> str:
    styler = df.style.hide(axis="index").format(
        {c: float_fmt for c in df.select_dtypes("float").columns}
    ).format_index(escape="latex", axis=1).format(escape="latex-math", subset=df.select_dtypes("object").columns)
    return styler.to_latex(hrules=True, caption=caption, label=label, position="t", position_float="centering")


## Config

In [ ]:
GRAN = sl.PAPER_GRANULARITY      # 0.05
MIN_GROUPS = sl.MIN_GROUPS       # 50
MIN_PURITY = sl.MIN_PURITY       # 0.99

## Load

~1 min: the 3.2 GB `alert_groups_raw.json` plus the CSV for the SignatureID
lookup (AlertGroup doesn't carry it -- ingest hashes it into `group_id`, so
`load_cscas_signature_lookup` rebuilds that key from the CSV).

In [ ]:
groups = load_or_build_alert_groups("cscas", sl.CSCAS_ALERT_GROUPS_CACHE_DIR)
groups.sort(key=lambda g: g.start_ts or "")
signature_ids = sl.signature_ids_for(groups, sl.load_cscas_signature_lookup())
print(f"{len(groups):,} groups, {len(np.unique(signature_ids))} signatures")

### Sanity: signatures are class-pure (Table 5.1)

In [ ]:
from thesis.experiments._shared import labels_and_mask

labels, mask = labels_and_mask(groups)
per_sig = pd.DataFrame({"sig": signature_ids[mask], "y": labels[mask]}).groupby("sig")["y"]
print("mixed signatures:", int((per_sig.min() != per_sig.max()).sum()), "of", per_sig.ngroups)

## §1 Purity table

In [ ]:
res = sl.run_purity_step(groups, signature_ids, gran=GRAN)
res["windows"]

**Reading the direction column.** At a ~1.3–1.5% base attack rate, any value
whose attack rate is ≤ 1% already clears purity 0.99 on the *benign* side, so
benign-pure flags are cheap (lift < 1 just means "fewer attacks than average").
The **attack-pure** values are the SignatureID-proxy risk; the benign-pure ones
mostly describe where the hard-to-find attacks hide (`share_attack_groups`).
`smpd_bin` (SignatureMatchesPerDay quantiles, in Base(5)) is diagnostic only.

In [ ]:
FLAG_COLS = ["field", "value", "n_groups", "n_attack", "attack_rate", "lift", "n_signatures", "direction"]

def flagged(table: pd.DataFrame, diagnostic: bool = False) -> pd.DataFrame:
    t = table[table["flagged"] & (table["diagnostic"] == diagnostic)]
    return t.sort_values(["direction", "n_groups"], ascending=[True, False])[FLAG_COLS]

for name in ("w0", "pooled"):
    print(f"\n=== {name}: flagged signature-derived values ===")
    display(flagged(res[f"{name}_table"]))
    print(f"=== {name}: coverage ===")
    display(res[f"{name}_coverage"])

### Diagnostic: SignatureMatchesPerDay

In [ ]:
for name in ("w0", "pooled"):
    t = res[f"{name}_table"]
    d = t[t["diagnostic"]]
    print(f"{name}: {int(d['flagged'].sum())}/{len(d)} smpd bins flagged, "
          f"{int((d['n_signatures'] == 1).sum())} backed by a single signature")

### Full table (all values, not only flagged)

In [ ]:
res["pooled_table"].query("not diagnostic").sort_values(["field", "n_groups"], ascending=[True, False])

## LaTeX

In [ ]:
t = flagged(res["pooled_table"]).rename(columns={
    "field": "Field", "value": "Value", "n_groups": "#groups", "n_attack": "#attack",
    "attack_rate": "Attack rate", "lift": "Lift", "n_signatures": "#sigs", "direction": "Pure side",
})
print(to_booktabs(t, "Near-pure signature-derived values (purity $\\geq$ 0.99, $\\geq$ 50 groups), pooled training windows.", "tab:sig-purity"))
print(to_booktabs(res["pooled_coverage"], "Coverage of flagged values, pooled training windows.", "tab:sig-purity-coverage"))

## §2–3 Walk-forward ablation

Arms: **(a)** Base(5), **(b)** Base(5) + mining on all items, **(c)** Base(5) +
mining without signature-derived items (`CSCAS_SIGNATURE_DERIVED_FIELDS`:
category, ruleset, cve_present, cve_age_years, proto_mismatch, qualifier_*).
RF class-weighted, 5 seeds (mined schema fixed per window, so seeds = classifier
variance). Seen = attack's SignatureID appeared in Wᵢ; *seen_prior* = appeared
anywhere before Wᵢ₊₁ (robustness). Seen/unseen recall is pooled over steps per
seed, then mean ± std over seeds.

In [ ]:
from pathlib import Path

from thesis.paths import ROOT, RESULTS_DIR

RUN_DIR = None  # None -> latest full run (results mirror), else a Path

def _latest_run() -> Path:
    for base in (RESULTS_DIR / "paper" / "signature-leakage",
                 ROOT / "artifacts" / "experiments" / "signature_leakage"):
        runs = sorted(d for d in base.glob("*") if (d / "ablation_summary.csv").exists()) if base.exists() else []
        if runs:
            return runs[-1]
    raise FileNotFoundError("no signature_leakage run found")

run_dir = Path(RUN_DIR) if RUN_DIR else _latest_run()
per_step = pd.read_csv(run_dir / "per_step_results.csv")
preds = pd.read_csv(run_dir / "predictions.csv.gz")
print(run_dir, "|", per_step.groupby("arm").size().to_dict(), "rows per arm")

### §3 Ablation table

In [ ]:
summary = sl.summarize_ablation(per_step)
cols = ["arm", "arm_label"] + [f"{m}_{s}" for m in sl.SUMMARY_METRICS + ["recall_seen", "recall_unseen", "recall_seen_prior", "recall_unseen_prior"] for s in ("mean", "std")]
summary[cols]

In [ ]:
def pm(df, metric, d=3):
    return df[f"{metric}_mean"].map(f"{{:.{d}f}}".format) + " ± " + df[f"{metric}_std"].map(f"{{:.{d}f}}".format)

paper = pd.DataFrame({
    "Arm": summary["arm_label"],
    "F1": pm(summary, "f1"), "PR-AUC": pm(summary, "pr_auc"), "FP/day": pm(summary, "fp_per_day", 1),
    "Recall (seen)": pm(summary, "recall_seen"), "Recall (unseen)": pm(summary, "recall_unseen"),
})
display(paper)
print(to_booktabs(paper.apply(lambda col: col.str.replace("±", r"$\pm$", regex=False)), "Walk-forward ablation (RF, 5 seeds, $\\gamma=0.05$): mean $\\pm$ std over seeds. "
      "Seen/unseen: attack groups whose signature did/did not appear in the training window.", "tab:sig-ablation"))

### §2 How many attacks are unseen, per step

In [ ]:
one = per_step[(per_step["arm"] == per_step["arm"].iloc[0]) & (per_step["seed"] == per_step["seed"].min())]
display(one[["step_i", "n_attack", "n_attack_seen", "n_attack_unseen", "n_attack_unseen_prior"]])
print("unseen share of attacks (pooled):", one["n_attack_unseen"].sum() / one["n_attack"].sum())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, part in zip(axes, ("seen", "unseen")):
    g = per_step.groupby(["arm", "step_i"])[f"recall_{part}"].mean().unstack(0)
    for arm in g.columns:
        ax.plot(g.index, g[arm], marker="o", ms=3, label=sl.ARM_LABELS[arm])
    ax.set_title(f"Recall on {part} signatures (mean over seeds)")
    ax.set_xlabel("step i (train Wᵢ → test Wᵢ₊₁)")
axes[0].set_ylabel("recall"); axes[0].legend(fontsize=8)
plt.tight_layout()

### Decision

- (b) beats (a) on unseen too, few attack-pure values → mining finds patterns
  beyond the signature.
- (b)'s gain mostly gone in (c) / on unseen → mining mainly compiles signature
  knowledge; narrow the claim and run the top-2 comparison methods in setting (c).

In [ ]:
s = summary.set_index("arm")
for metric in ("f1", "pr_auc", "recall_unseen", "recall_seen"):
    if {"a", "b", "c"} <= set(s.index):
        gain_b = s.loc["b", f"{metric}_mean"] - s.loc["a", f"{metric}_mean"]
        gain_c = s.loc["c", f"{metric}_mean"] - s.loc["a", f"{metric}_mean"]
        kept = f"{gain_c / gain_b:.0%}" if gain_b > 0 else "n/a (b does not beat a)"
        print(f"{metric:15s} gain b-a={gain_b:+.3f}  c-a={gain_c:+.3f}  share of gain kept without sig items={kept}")